In [1]:
import pandas as pd
import pyarrow.parquet as pq

def build_split_indexes(cumc_split_path: str):
    # Read only needed cols, and use compact dtypes
    split = pd.read_csv(
        cumc_split_path,
        usecols=["subject_id", "split_name"],
        dtype={"subject_id": "int64", "split_name": "category"},
    )
    train_idx = pd.Index(split.loc[split["split_name"].eq("train"), "subject_id"].values)
    test_idx  = pd.Index(split.loc[split["split_name"].eq("test"),  "subject_id"].values)
    return train_idx, test_idx

def find_not_belong_subject_ids(
    parquet_path: str,
    allowed_idx: pd.Index,
    batch_size: int = 2_000_000,
):
    """
    Streams subject_id from parquet and returns a set of subject_ids not in allowed_idx.
    If the mismatch set could be huge, you can modify this to write to disk instead.
    """
    pf = pq.ParquetFile(parquet_path)
    not_belong = set()

    for batch in pf.iter_batches(columns=["subject_id"], batch_size=batch_size):
        arr = batch.column(0).to_numpy(zero_copy_only=False)

        # Fast membership: get_indexer returns -1 when not found
        missing = arr[allowed_idx.get_indexer(arr) == -1]

        # Dedup within-batch before adding to python set (faster & smaller)
        if missing.size:
            not_belong.update(pd.unique(missing))

    return not_belong

def check_cohort_splits(train_path: str, tuning_path: str, held_path: str, cumc_split_path: str):
    train_idx, test_idx = build_split_indexes(cumc_split_path)

    bad_train  = find_not_belong_subject_ids(train_path,  train_idx)
    bad_tuning = find_not_belong_subject_ids(tuning_path, train_idx)
    bad_held   = find_not_belong_subject_ids(held_path,   test_idx)

    def report(name, bad_set, show=50):
        if not bad_set:
            print(f"[OK] {name}: all subject_ids belong to expected standard split.")
        else:
            bad_list = sorted(bad_set)
            print(f"[BAD] {name}: {len(bad_list)} subject_ids do NOT belong.")
            print(f"  First {min(show, len(bad_list))}: {bad_list[:show]}")

    report("train.parquet (expected standard TRAIN)", bad_train)
    report("tuning.parquet (expected standard TRAIN)", bad_tuning)
    report("held_out.parquet (expected standard TEST)", bad_held)

    # If you truly need *all* IDs printed (can be enormous), uncomment:
    # print("All bad train IDs:", sorted(bad_train))
    # print("All bad tuning IDs:", sorted(bad_tuning))
    # print("All bad held IDs:", sorted(bad_held))

    return bad_train, bad_tuning, bad_held

# Example usage:
cumc_split_path = "/data2/processed_datasets/zj2398/femr/cumc/mtpp_vocab_all/cumc_split.csv"
for disease in ["MASLD"]:
    train_path =  f"/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/{disease}/train.parquet"
    tuning_path =  f"/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/{disease}/tuning.parquet"
    held_path =  f"/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/{disease}/held_out.parquet"
    bad_train, bad_tuning, bad_held = check_cohort_splits(train_path, tuning_path, held_path, cumc_split_path)


[OK] train.parquet (expected standard TRAIN): all subject_ids belong to expected standard split.
[OK] tuning.parquet (expected standard TRAIN): all subject_ids belong to expected standard split.
[OK] held_out.parquet (expected standard TEST): all subject_ids belong to expected standard split.
